# Primal Linear Programming Problems
Suppose you have a _linear_ objective function $O:\mathbb{R}^{n}\to\mathbb{R}$ of the continuous decision variable vector $\mathbf{x}\in\mathbb{R}^{n}$ whose values are constrained by a system of $m$ linear inequalities. To calculate an optimal value of the decision variable vector $\mathbf{x}$, we can formulate the problem as a _primal linear programming_ problem:
$$
\begin{align*}
\text{maximize} &\, \sum_{i=1}^{n} c_{i}\cdot{x}_{i}\\
\text{subject to}~\mathbf{A}\cdot\mathbf{x} &\leq \mathbf{b}\quad\mathbf{A}\in\mathbb{R}^{m\times{n}}\,\text{and}\,\mathbf{b}\in\mathbb{R}^{m}\\
~x_{i}&\geq {0}\qquad{i=1,2,\dots,n}
\end{align*}
$$
where $c_{i}\in\mathbb{R}$ are the coefficients of the objective function, $x_{i}\in\mathbb{R}$ are the decision variables, and $\mathbf{A}$ and $\mathbf{b}$ are the constraint matrix and right-hand side vector, respectively. The goal is to maximize the objective function while satisfying the constraints.

Let's look at two well-known techniques for solving primal linear programming problems: the Simplex method and Interior Point methods.
___

## Revised Simplex Algorithm
The original Simplex Method, invented by George Dantzig in 1947, is an iterative algorithm that solves a linear program by moving around the collection of corner points of the feasible _polytope_, improving the objective at each step until no further gain is possible. 

> __Myth or fact?__ As a graduate student in 1939, Dantzig once arrived late to a statistics lecture, mistook two well-known unsolved problems on the blackboard for homework, and solved them over the next few days, before realizing they were open research questions! That story is true, but it happened years before he developed the simplex method and did not directly inspire the algorithm. But still, fun story - being late isn't always bad!

__Simplex is a big deal__: Before simplex, LPs were mostly a theoretical curiosity; afterwards, they became essential tools, radically improving resource allocation and strategic planning in the second half of the twentieth century. Over seventy years later, the simplex method remains widely used in commercial optimization. This is despite some not-so-great worst-case performance bounds!

### Algorithm
We are going to focus on the _revised simplex algorithm_, developed by Dantzig and coworkers in the 1950s at the RAND Corporation, which is a more efficient version of the original method. 

The __key idea__ behind the revised simplex algorithm is to partition the decision variables into a _basic set_ and a _non-basic set_. Then, we iteratively add and subtract variables from these sets and estimate their values, iteratively improving the objective function until we reach an optimal solution.
* A _basic variable_ is one you’re allowing to _turn on_ i.e., take a non-zero value. In the simplex method, you swap which variables are on or off to move from one corner of the feasible region to the next.
* A _nonbasic variable_ is one you keep _off_ (held at zero) — think of non-basic variables as benchwarmers not in play. When it looks promising, you swap it in (make it basic) to move to a potentially better solution.
* _How are the basic (and non-basic) variables related to corners?_ At any corner of the feasible polytope, exactly $m$ linearly independent variables are _turned on_ (basic set) so they solve the $m$ active equality constraints, and all other variables (non-basic set) are zero. Choosing which $m$ variables are basic (and solving for them) picks out one specific corner of the polytope, while every non-basic variable being zero defines the edges that meet at that corner.

Let's sketch out the revised simplex algorithm:

__Initialization__: Given a linear program of the form: $\min\left\{c^\top x\mid Ax+s = {b},\;x\ge0,\; s\ge0,\;b\ge0\right\}$ where $A\in\mathbb{R}^{m\times{n}}$, $b\in\mathbb{R}^{m}$, and $c\in\mathbb{R}^{n}$, we want to find the optimal solution $x^{\star}$ that minimizes the objective function $c^\top x$ subject to the constraints defined by $Ax + s = b$ and the non-negativity conditions on $x$ and $s$. Note: We assume the LP has been converted from the maximization form above by negating the objective coefficients.

Let $z = \left(x,s\right)\in\mathbb{R}^{n+m}$. Define the initial Basic set $B=\left\{s_{1},s_{2},\dots,s_{m}\right\}$, and the Non-Basic set $N=\left\{x_{1},x_{2},\dots,x_{n}\right\}$, where $x^{(0)}=0$ and $s^{(0)}=b$. Set the iteration counter $t\gets{0}$, and the maximum number of iterations $T$. Set $\texttt{converged}\gets\texttt{false}$.

While not $\texttt{converged}$ __do__:
1. __Optimality test__. Compute the reduced cost $\mu_{i} = c_{i} - \lambda^{\top}A_{i}$ for each non-basic variable $i \in N$, where $\lambda = (A_B^T)^{-1}c_B$ are the dual multipliers and $A_B$ is the $m \times m$ basic matrix formed by columns of $A$ corresponding to basic variables.
    - If $\mu_{i} \geq 0$ for all $i \in N$ then set $\texttt{converged}\gets\texttt{true}$ and return the current solution $z^{(t)}$.
    - If $\mu_{i} < 0$ for any $i\in{N}$, the current solution is __not optimal__; there is a non-basic variable that, if _turned on_, will strictly decrease the objective.
2. __Direction and ratio test__. Select $e \gets \arg\min_{i \in N} \mu_{i}$. This variable $e$ will enter the basis (i.e., be turned on) to improve the objective. Compute the direction $\mathbf{d} = A^{-1}_{B}A_{e}$, where $A_{e}$ is the column of the constraint matrix corresponding to the variable $e$, and $A_{B}$ is the submatrix of $A$ formed by the basic variables.
    - If $\left\{j \mid d_{j} > 0\right\} = \emptyset$: the current solution is __unbounded__. Exit the algorithm with an __error__.
    - Compute the step size $\alpha = \min\left\{\frac{(z_B)_{j}}{d_{j}}\mid d_{j} > 0\right\}$, where $(z_B)_j$ is the jth element of the basic variable vector.
    - Compute the index of the variable that will _leave_ the basic set: $l = \arg\min\left\{\frac{(z_B)_{j}}{d_{j}}\mid d_{j} > 0\right\}$.
3. __Pivot and update__: Update the basic and non-basic sets: $B \gets \left(B \setminus \{B_{l}\}\right)\cup \{e\}$ and $N \gets \left(N \setminus \{e\}\right) \cup \{B_l\}$. This means we swap the entering variable $e$ into the basic set and the leaving variable $B_l$ into the non-basic set.
    - Set $z_{e} \gets \alpha$ and update the solution vector $z_{B}^{(t+1)} \gets z_{B}^{(t)} - \alpha \cdot\mathbf{d}$.
    - Update the dual multipliers $\lambda\gets\left(A^{\top}_{B}\right)^{-1}c_{B}$, and the iteration counter $t \gets t + 1$.
4. __Check convergence__: If $t \geq T$, set $\texttt{converged}\gets\texttt{true}$ and return the current solution $z^{(t)}$. Exit the algorithm with an __error__ if the maximum number of iterations is reached without convergence. Otherwise, loop back to step 1.

Wow! That seems intense. How efficient is the simplex algorithm? 
* In the __worst case__, the simplex method can take _exponential time_ in the number of variables—Klee and Minty’s 1972 example shows it may visit all $2^n$ vertices of an $n$-dimensional cube, forcing on the order of $2^n$ pivots. Thus, it has $O(2^n)$ worst-case complexity.
* However, __in practice__, the simplex method is often very efficient. It performs well on most real-world problems, and its average-case performance is polynomial time for many practical instances. The worst-case exponential bound is rarely encountered in practice, as most LPs have a structure that allows the simplex method to converge quickly.

Next, let's examine the second class of algorithms based on the KKT conditions: Interior Point methods.

___

## Interior-Point Methods
Interior-point methods solve a linear program by traversing the _interior_ of the feasible region—avoiding the corners until the very end—using a _barrier function_ to enforce the inequalities. They trace a path defined by modified KKT conditions (with logarithmic penalties) and converge to the optimum in a small, predictable number of steps.
* _Barrier function?_ An invisible wall that shoots to infinity as you approach any constraint boundary, repelling your iterates and keeping them safely in the interior while still guiding you toward the optimum. Gradually lowering its weight lets the solution creep closer to the boundary without crossing it.
* _Logarithmic penalties?_ An example barrier function. Logarithmic penalties are terms of the form $-\mu\ln(s)$ added to your objective, because $\ln(s)\to-\infty$ as $s\to0^+$, they impose a _huge cost_ for getting too close to a constraint boundary.  As you decrease the weight $\mu$, you gradually lessen the penalty’s influence, letting the solution drift closer to the true feasible region edge.

__The big idea__: Interior-point methods navigate the feasible region's interior using a smooth barrier that dominates at edges, taking Newton-style steps along a central path to the optimum. Unlike the simplex method, which moves along the boundary and zig-zags across facets, these algorithms follow a direct, well-conditioned route through the interior, offering more predictable performance on large or ill-conditioned problems.


### Algorithm
Let's sketch out an interior point algorithm to solve an LP.

__Initialization__: Given a linear program of the form: $\min\left\{c^\top x\mid Ax+s = {b},\;x\ge0,\; s\ge0,\;b\ge0\right\}$ where $A\in\mathbb{R}^{m\times{n}}$. Specify an _initial strictly feasible guess_ for ($x^{(0)}, s^{(0)}, \lambda^{(0)}, \nu^{(0)})$ (more on this later). Specify a tolerance $\epsilon>0$, a maximum number of iterations $T$, an iteration counter $t\gets{0}$, set $\texttt{converged}\gets\texttt{false}$ and choose a _reduction factor_ $\sigma\in(0,1)$. Finally, compute $\mu^{(0)}$:
$$
\begin{align*}
\mu^{(0)} \gets \frac{x^{(0)^{\top}}\nu^{(0)} + s^{(0)^{\top}}\lambda^{(0)}}{n+m}
\end{align*}
$$

While not $\texttt{converged}$ __do__:
1. Compute the four residuals ($r_{P}, r_{D}, r^{x}_{C}, r^{s}_{C}$):
     - _Primal residual_: $r_{P}\gets{Ax^{(t)} +s^{(t)} - b}$. The primal residual $r_{P}$ shows how much your guess $(x,s)$ violates constraints. Each entry of $r_P$ is the gap between your decision, slack variables, and the right-hand side $b$; zero indicates your solution is feasible.
     - _Dual residual_: $r_D \gets A^{\top}\lambda^{(t)}+\nu^{(t)} - c$. The dual residual $r_{D}$ indicates how much your dual variables $(\lambda,\nu)$ deviate from the stationarity condition of the Lagrangian. A nonzero $r_D$ shows you need to adjust the multipliers or primal $x$ to move toward optimality.
     - _Complementarity residual_: $r^{x}_{C}\gets X^{(t)}\nu^{(t)} - \mu^{(t)}\mathbf{1}$, where $X = \text{diag}(x)$. The complementarity residual $r^{x}_{C}$ measures deviation from the ideal central product $x_j \nu_j = \mu$ for each coordinate. When $r_C^x=0$, each $x_j$ and its dual partner $\nu_j$ satisfy the perturbed complementary-slackness condition, placing you on the central path between primal and dual feasibility.
     - _Slack-complementarity residual_: $r^{s}_{C} \gets S^{(t)}\lambda^{(t)} - \mu^{(t)}\mathbf{1}$, where $S=\text{diag}(s)$. The slack-complementarity residual $r^{s}_{C}$ measures how far each slack variable $s_i$ and its dual multiplier $\lambda_i$ are from satisfying the central-path condition. When $r_C^s = 0$, each slack $s_{i}$ and multiplier $\lambda_{i}$ satisfy $s_{i}\lambda_{i} = \mu$.

2. Compute the Jacobian matrix $J$:
$$
     J =
     \begin{pmatrix}
       A & I & 0 & 0\\
       0 & 0 & A^\top & I\\
       \text{diag}(\nu^{(t)}) & 0 & 0 & \text{diag}(x^{(t)})\\
       0 & \text{diag}(\lambda^{(t)}) & \text{diag}(s^{(t)}) & 0
     \end{pmatrix}.
   $$
3. Take a _Newton step_: solve for the updates $\Delta x, \Delta s, \Delta \lambda, \Delta \nu$:
$$
     J\,
     \begin{pmatrix}
       \Delta x\\[3pt]\Delta s\\[3pt]\Delta\lambda\\[3pt]\Delta\nu
     \end{pmatrix}
     = -\,
     \begin{pmatrix}
       r_P\\r_D\\r_C^x\\r_C^s
     \end{pmatrix}.
   $$

4. Choose the step size $\alpha$: Choose the largest $\alpha\in(0,1]$ such that:
   $$
     x^{(t)} + \alpha\,\Delta x > 0,\quad
     s^{(t)} + \alpha\,\Delta s > 0,\quad
     \lambda^{(t)} + \alpha\,\Delta\lambda > 0,\quad
     \nu^{(t)} + \alpha\,\Delta\nu > 0.
   $$
5. Update the system solution ($x,s,\lambda,\nu,\mu$):
 - Set $x^{(t+1)} \gets x^{(t)} + \alpha\,\Delta x$.
 - Set $s^{(t+1)} \gets s^{(t)} + \alpha\,\Delta s$.
 - Set $\lambda^{(t+1)} \gets \lambda^{(t)} + \alpha\,\Delta\lambda$
 - Set $\nu^{(t+1)} \gets \nu^{(t)} + \alpha\,\Delta\nu$
 - Set $\mu^{(t+1)} \gets \sigma\,\mu^{(t)}$

6. Check for convergence. Update the iteration counter $t\gets{t+1}$.
   - If $\mu^{(t)} \leq \epsilon$ (or $\|F\| \leq \epsilon$), then $(x^{(t)},s^{(t)})$ approximates the true optimum. Set $\texttt{converged}\gets\texttt{true}$, return $(x^{(t)},s^{(t)})$. Here, $F$ is the stacked residual vector.
   - If $t>T$, we've run out of iterations. Set $\texttt{converged}\gets\texttt{true}$, return $(x^{(t)},s^{(t)})$, which is the solution we have so far.

**Relation to KKT**: At each $\mu>0$, the algorithm solves the **perturbed** KKT system
$\;r_P=0,\;r_D=0,\;r_C^x=0,\;r_C^s=0$
which approaches the **exact** (unperturbed) KKT conditions as $\mu\to0$

___